# DO SOFT DRINKS SOLD IN THE UK AND FRANCE CONTAIN LESS SUGAR THAN THOSE SOLD IN GERMANY AND ITALY?

## Introduction

The goal of this project is to check whether soft drinks sold in countries with a sugar tax contain less sugar than those sold in countries without one. The idea is that a tax gives manufacturers a financial reason to reformulate their drinks, so a market where sugary drinks are taxed should end up with less sugar on the shelves. The UK and France both tax sugary soft drinks, while Germany and Italy do not, so comparing the sugar content of sodas across these four countries should show whether the policy leaves a mark on the products themselves.

In [9]:
import json
from pathlib import Path

import pandas as pd
import requests

In [10]:
# Used capital letters to clarify that these variables will not be changed throughout the project. Fixed Variables. 
URL = "https://world.openfoodfacts.org/api/v2/search"
HEADERS = {"User-Agent": "ME204-LSE-project/1.0 j.milagro-caro@lse.ac.uk"}

## The API

The data for this project comes from the Open Food Facts API. Open Food Facts is a free, open database of food products from around the world, built by volunteers who scan barcodes and upload the information from the packaging. Each product in the database has a barcode, a name, a brand, a list of countries where it is sold, and a set of nutritional values taken from the label. The API is free to use and does not need a key or an account for reading data, which is why it works well for a project like this one.

### How I found the right endpoint

- I started from the API documentation homepage, which lists the different versions of the API and explains what each one can do.
- The documentation has a table comparing versions, and it shows that structured search (filtering by things like category and country) is only available in version 2, at `/api/v2/search`. Version 3 is the newer one, but it cannot search.
- The reference pages showed me the names of the filters I needed: `categories_tags_en` to pick a food category, and `countries_tags_en` to pick a country.
- The same pages explained `fields`, which lets me ask for only the columns I need instead of the whole product record, and `page_size`, which controls how many products come back at once.
- The Authentication section asks every user to send a custom User-Agent header with an app name and a contact email, so I added that to my requests.
- I checked that my category tag was spelled correctly by opening `world.openfoodfacts.org/category/sodas` in a browser and seeing that products appeared.

### Limitations of the API

- There are rate limits. Search requests are capped at 10 per minute, so I had to pause between calls to avoid being blocked.
- Each request returns a limited number of products, so getting a full category means requesting several pages one after another rather than everything at once.
- The version I need for searching, version 2, is marked as deprecated in the documentation. It still works, but it may not be supported forever.
- The API has no full-text search, so I could only filter by exact tag names from their taxonomy. If I spell a tag wrong, I get an empty result rather than an error message.
- The data is added by volunteers, so fields are often incomplete and the documentation itself warns that there is no guarantee the data is accurate or complete.
- The documentation asks users not to pull more than a few hundred products through the API, and to download the full dataset as a file instead. This puts a practical ceiling on how much data I can collect this way.

In [11]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

# Store the countries we will be using in the API as a list
COUNTRIES = ["united-kingdom", "france", "germany", "italy"]

In [12]:
def collect_sodas(country, data_folder=RAW_DIR):
    """Fetch sodas for one country and save the raw JSON."""
    params = {
        "categories_tags_en": "sodas",
        "countries_tags_en": country,
        "fields": "code,product_name,brands,nutriments,nutriscore_grade",
        "page_size": 100,
    }

    response = requests.get(URL, params=params, headers=HEADERS, timeout=60)

    if response.status_code != 200:
        return f"Error: {response.status_code} for {country}"

    path = data_folder / f"{country}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(response.json(), f, indent=2)

    return "success"

for country in COUNTRIES: 
    print(country, collect_sodas(country))

united-kingdom Error: 503 for united-kingdom
france success
germany success
italy Error: 503 for italy


### A change of endpoint

The Open Food Facts documentation recommends the `/api/v2/search` endpoint for
filtering products by category and country, so that is where I started. Every
request to it returned a 503 error. I checked the same request in a browser and
confirmed it was correctly formed, which ruled out a mistake in my code, and the
error page indicated the service was under heavy load and deprioritising
anonymous requests.

Searching for the problem showed that other developers had reported the same
error on this endpoint earlier in the year, and that the older search backend is
being retired in favour of a newer service, Search-a-licious, hosted at
`search.openfoodfacts.org`. This service is mentioned in the official
documentation, though only in relation to full-text search rather than as a
replacement for the endpoint I was using.

I tested the new endpoint in a browser before changing any code, confirmed it
returned both products and nutritional values, and then rewrote my collection
function to use it. The structure of the pipeline did not change: only the URL
and the way the filters are written.

In [13]:
URL = "https://search.openfoodfacts.org/search"

In [14]:
def collect_sodas(country, data_folder=RAW_DIR):
    """Fetch sodas for one country and save the raw JSON."""
    data_folder = Path(data_folder)
    data_folder.mkdir(parents=True, exist_ok=True)

    params = {
        "q": f'categories_tags:"en:sodas" AND countries_tags:"en:{country}"',
        "fields": "code,product_name,brands,nutriments",
        "page_size": 100,
    }

    response = requests.get(URL, params=params, headers=HEADERS, timeout=60)

    if response.status_code != 200:
        return f"Error: {response.status_code} for {country}"

    path = data_folder / f"{country}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(response.json(), f, indent=2)

    return "success"

for country in COUNTRIES: 
    print(country, collect_sodas(country))

united-kingdom success
france success
germany success
italy success


In [15]:
# This cell of code helps me identify how the JSON is structred in order to create my DataFrame later on. 

with open(RAW_DIR / "france.json", encoding="utf-8") as f:
    country = json.load(f)

print(json.dumps(country["hits"][0], indent=2))

{
  "code": "5449000285720",
  "brands": [
    "Fanta"
  ],
  "nutriments": {
    "energy-kcal_100g": 5,
    "carbohydrates_100g": 0,
    "proteins_100g": 0,
    "sugars_100g": 0,
    "saturated-fat_100g": 0,
    "fat_100g": 0
  },
  "product_name": "Fanta Rasberry"
}


## Variables that will be kept in the dataframe

The API returns a lot of information about each product, but only a few fields are needed to answer the question. The API returns a lot of information about each product, but only a few fields are
needed to answer the question. 

- **`code`** — the product barcode. This is the unique identifier for each product, so it lets me check for duplicates and confirm that the same drink has not been counted twice.
- **`product_name`** — the name of the product. Not used in the analysis itself, but needed to see what is actually in the sample and to spot records that are empty or clearly not soft drinks.
- **`brands`** — the brand that makes the product. Useful for checking whether the same brands appear across all four countries, and for spotting whether one brand dominates a particular country.
- **`sugars_100g`** — the amount of sugar in grams per 100g of product. This is the variable the whole question rests on, and it comes from inside the `nutriments` section of each product record.
- **`country`** — which country the product was collected for. This does not come from the API response itself, since each request already filters to one country. I add it when building the dataframe, taking it from the file each set of products was saved in.

Everything else the API offers, such as ingredients, packaging, images and contributor history, is left out because it does not help answer this particular question.

Two of these columns are often missing. Some products have no name, and some have no sugar value at all, because the volunteer who added them never filled that information in. These are kept in the table rather than removed at this stage, so that the amount of missing data can be counted per country before deciding what to do about it.